# ECON326 - Final Project

## Import Statements

In [ ]:
install.packages("modelsummary")
install.packages("sandwich") 
install.packages("kableExtra")
install.packages("estimatr")
library(tidyverse)
library(haven)
library(dplyr)
library(scales)
library(stargazer)
library(car)
library(lmtest) 
library(sandwich) 
library(modelsummary)
library(knitr)
library(kableExtra)
library(estimatr)

census_data <- read_csv("data/data_donnees_2021_ind_v2.csv")

In [ ]:
options(modelsummary_factory_default = "latex_tabular")

In [ ]:
head(census_data)

## Data Filtering & Cleaning

In [ ]:
# includes only variables needed for our regression
census_data <- census_data |>
    select(TotInc, POB, HDGREE, CIP2021, LOCSTUD, NOC21, KOL, AGEIMM, Gender, MarStH, HHSIZE)
head(census_data)

In [ ]:
# removing citizens with unavailable immigrant data
census_data <- census_data |>
    filter(AGEIMM < 88 & TotInc > 0 & TotInc < 1e6 & HDGREE < 20)
head(census_data)

In [ ]:
# checking for extreme values of TotInc
summary(census_data$TotInc)

In [ ]:
# Converts integers into categories as strings (words), set up as factor variables
census_data <- census_data |>
    mutate(Gender = factor(Gender, levels = c(1,2), labels = c("Female", "Male")),
          POB = factor(POB, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,88),
                       labels = c("Canada", "USA", "Central America", "Jamaica", "Caribbean", "South America", "UK", "Germany", "France", "Northern + Western Europe", "Poland", "Eastern Europe", 
                                  "Italy", "Portugal", "Southern Europe", "Eastern Africa", "Northern Africa", "South + West Africa", "Iran", "Middle East + West Central Asia", "China", "Hong Kong", 
                                  "South Korea", "East Asia", "Philippines", "Vietnam", "Southeast Asia", "India", "Pakistan", "Sri Lanka", "Other South Asia", "Oceania + others", "N/A")),
          MarStH = factor(MarStH, levels = c(1,2,3,4,5,6), labels = c("Never married", "Married", "Living common law", "Separated", "Divorced", "Widowed")),
          CIP2021 = relevel(factor(CIP2021, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,88,99),
                          labels = c("Education", "Visual + Performing Arts", "Humanities", "Social Sciences + Law", "Business + Public Admin", "Physical + Life Sciences", "Math + Technology", 
                                     "Architecture + Engineering", "Agriculture", "Health", "Services", "Other", "No certificate, diploma, or degree", "N/A", "N/A")),
                            ref = "No certificate, diploma, or degree"),
          LOCSTUD = factor(LOCSTUD, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,99),
                          labels = c("Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", "Canada", 
                                     "US", "Other Americas", "Europe", "East Asia", "South + Southeast Asia", "Other", "N/A")),
          AGEIMM = relevel(factor(AGEIMM, levels= c(1,2,3,4,5,6,7,8,9,10,11,12,13,88,99),
                         labels = c("0 to 4 yo", "5 to 9 yo", "10 to 14 yo", "15 to 19 yo", "20 to 24 yo", "25 to 29 yo", "30 to 34 yo", 
                                    "35 to 39 yo", "40 to 44 yo", "45 to 49 yo", "50 to 54 yo", "55 to 59 yo", "60+ yo", "N/A", "N/A")),
                          ref = "N/A"),
          HDGREE = relevel(factor(HDGREE, levels = c(1,2,3,4,5,6,7,8,9,10,11,12,13,88,99),
                                 labels = c("No certificate, diploma, or degree",
                                           "High school diploma",
                                           "Non-apprenticeship Trades",
                                           "Apprenticeship Certificate",
                                           "3 months to 1 year in postsecondary",
                                           "1 to 2 years in postsecondary",
                                           "More than 2 years in postsecondary",
                                           "Diploma below bachelor's",
                                           "Bachelor's Degree",
                                            "University diplom above Bachelor's",
                                           "Degree in medicine or related",
                                           "Master's Degree",
                                           "Earned Doctorate",
                                           "N/A",
                                           "N/A")),
                                  ref = "No certificate, diploma, or degree")) |>
    mutate(
        noc21 = case_when(
            NOC21 == 1 ~ "Management + Business",
            NOC21 == 2 ~ "Management + Business",
            NOC21 == 3 ~ "Management + Business",
            NOC21 == 4 ~ "Management + Business",
            NOC21 == 5 ~ "Management + Business",
            NOC21 == 6 ~ "Management + Business",
            NOC21 == 7 ~ "Natural + Applied Sci",
            NOC21 == 8 ~ "Natural + Applied Sci",
            NOC21 == 9 ~ "Health",
            NOC21 == 10 ~ "Health",
            NOC21 == 11 ~ "Health",
            NOC21 == 12 ~ "Educ + Law + Social + Govt Services",
            NOC21 == 13 ~ "Educ + Law + Social + Govt Services",
            NOC21 == 14 ~ "Educ + Law + Social + Govt Services",
            NOC21 == 15 ~ "Arts + Culture + Recreation",
            NOC21 == 16 ~ "Arts + Culture + Recreation",
            NOC21 == 17 ~ "Sales + Services",
            NOC21 == 18 ~ "Sales + Services",
            NOC21 == 19 ~ "Sales + Services",
            NOC21 == 20 ~ "Sales + Services",
            NOC21 == 21 ~ "Trades + Transportation",
            NOC21 == 22 ~ "Trades + Transportation",
            NOC21 == 23 ~ "Trades + Transportation",
            NOC21 == 24 ~ "Trades + Transportation",
            NOC21 == 25 ~ "Natural Resources + Production",
            NOC21 == 26 ~ "Natural Resources + Production",
            NOC21 == 88 ~ "N/A",
            NOC21 == 99 ~ "N/A",
        )) |>
    mutate(noc21 = relevel(as_factor(noc21), ref = "Sales + Services"))
head(census_data)

In [ ]:
# checking for multicollinearity through VIF test
vif_model_simple <- lm(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE, data = census_data)
vif(vif_model_simple, type = "predictor")

## Summary Statistics

In [ ]:
# Creates summary for each categorical variable,
# these summaries are included in the appendix 

# Create output folder if it doesn't exist
if (!dir.exists("tables")) dir.create("tables")

# List of categorical variables
cat_vars <- c("POB", "HDGREE", "CIP2021", "LOCSTUD", "noc21", "AGEIMM", "Gender", "MarStH", "KOL", "HHSIZE")

# Loop over each variable to create and export frequency tables
for (var in cat_vars) {
  freq_table <- census_data %>%
    count(!!sym(var)) %>%
    mutate(Percent = round(100 * n / sum(n), 2))

  # Rename columns for display
  colnames(freq_table) <- c("Category", "Count", "Percent")

  # Create LaTeX table and export it
  kbl(freq_table, format = "latex", booktabs = TRUE,
      caption = paste("Frequency Table for", var),
      label = paste0("tab:", var)) %>%
    kable_styling(latex_options = c("hold_position")) %>%
    save_kable(file = paste0("tables/", var, "_summary.tex"))
}

In [ ]:
# for continuous variables, just re-run again if run into warning message
continuous_variables_table <- datasummary(TotInc ~ Mean + SD + Min + Max + N,
                             data = census_data,
                             output = "latex_tabular",
                             title = "Numerical Variable Summary")

writeLines(as.character(continuous_variables_table), "tables/numeric_summary.tex")

## Model Explanation

In [ ]:
# only include immigrants in our model
immigrants_only <- subset(census_data, POB != "Canada")

# original model
RegImmigrants <- lm(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(RegImmigrants)
RegImmigrants_robust <- lm_robust(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(RegImmigrants_robust)

In [ ]:
# model #2, with NOC21 instead of simplified noc21 variable
RegImmigrantsNOC <- lm(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(RegImmigrantsNOC)

RegImmigrantsNOC_robust <- lm_robust(TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")
summary(RegImmigrants_robust)

In [ ]:
# transforming our regression into a log-linear model to eliminate total income's right-skewed distribution

immigrants_only$log_TotInc <- log(census_data$TotInc)

# model #3, the model we ultimately used for our analysis in the discussion and table of results
linlogRegImmigrants <- lm(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
    
summary(linlogRegImmigrants)

In [ ]:
# a robust test to eliminate heteroskedasticity
linlogRegImmigrants_robust <- lm_robust(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + noc21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")
summary(linlogRegImmigrants_robust)

In [ ]:
# model 4, used NOC21 variable instead of simplified noc21.
linlogRegImmigrantsNOC <- lm(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only)
summary(linlogRegImmigrantsNOC)

linlogRegImmigrantsNOC_robust <- lm_robust(log_TotInc ~ POB + HDGREE + CIP2021 + LOCSTUD + NOC21 + KOL + AGEIMM + Gender + MarStH + HHSIZE + POB:HDGREE + AGEIMM:HDGREE + POB:LOCSTUD, data = immigrants_only, se_type = "HC1")
summary(linlogRegImmigrantsNOC_robust)

## Table of Results

In [ ]:
# we use model summary as it is more flexible and easier to include robust regressions 
# to find significant results.

# model 1:
modelsummary(
    RegImmigrants_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

In [ ]:
# model 2:
modelsummary(
    RegImmigrantsNOC_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

In [ ]:
# shows summary including significant results and their levels of significance

# model 3, the model we used:
modelsummary(
    linlogRegImmigrants_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)

In [ ]:
# model 4:
modelsummary(
    linlogRegImmigrantsNOC_robust,
    title = "Income Differences by Place of Birth",
    stars = TRUE,
    statistic = "({std.error})",
    output = "latex_tabular",
    file = "linlogImmigrantsRobustReg_table.tex"
)